# 03 — Data Validation & Cleaning (Bab 3.4)Notebook ini mengimplementasikan proses validasi dan pembersihan data sesuai **6 dimensi kualitas data** yang didefinisikan di Bab 3.4.

## 3.1 Inisialisasi SparkSession

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, when, count, lit, sum as spark_sum

spark = SparkSession.builder \
    .appName("03_DataValidation") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "1g") \
    .config("spark.driver.memory", "1g") \
    .getOrCreate()

print(f"✅ Spark {spark.version} connected")

## 3.2 Load Data dari CSV

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("Transaction_ID", IntegerType(), False),
    StructField("Date", StringType(), False),
    StructField("Customer_ID", StringType(), False),
    StructField("Gender", StringType(), False),
    StructField("Age", IntegerType(), False),
    StructField("Product_Category", StringType(), False),
    StructField("Quantity", IntegerType(), False),
    StructField("Price_per_Unit", IntegerType(), False),
    StructField("Total_Amount", IntegerType(), False)
])

df_raw = spark.read.csv("/data/retail_sales_dataset.csv", header=True, schema=schema)
print(f"✅ Loaded {df_raw.count()} rows")

## 3.3 Dimensi 1: Kelengkapan (*Completeness*)Cek apakah ada nilai NULL di setiap kolom.

In [ ]:
print("=== NULL CHECK (Kelengkapan Data) ===")
null_counts = df_raw.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in df_raw.columns]
)
null_counts.show(truncate=False)

total_nulls = sum([null_counts.collect()[0][c] for c in df_raw.columns])
if total_nulls == 0:
    print("✅ PASS — Tidak ada nilai NULL. Kelengkapan 100%")
else:
    print(f"❌ FAIL — Ditemukan {total_nulls} nilai NULL")

## 3.4 Dimensi 2: Konsistensi (*Consistency*)Cek konsistensi format dan nilai kategorikal.

In [ ]:
print("=== KONSISTENSI FORMAT ===")

# Cek nilai unik Gender
print("\nGender values:")
df_raw.select("Gender").distinct().show()

# Cek nilai unik Product Category
print("Product Category values:")
df_raw.select("Product_Category").distinct().show()

# Validasi: hanya "Male"/"Female"
valid_genders = ["Male", "Female"]
invalid_gender = df_raw.filter(~col("Gender").isin(valid_genders)).count()

# Validasi: hanya 3 kategori
valid_categories = ["Beauty", "Clothing", "Electronics"]
invalid_category = df_raw.filter(~col("Product_Category").isin(valid_categories)).count()

print(f"Invalid Gender rows    : {invalid_gender} {'✅ PASS' if invalid_gender == 0 else '❌ FAIL'}")
print(f"Invalid Category rows  : {invalid_category} {'✅ PASS' if invalid_category == 0 else '❌ FAIL'}")

## 3.5 Dimensi 3: Akurasi (*Accuracy*)Validasi: `Total_Amount == Quantity × Price_per_Unit`

In [ ]:
print("=== VALIDASI BUSINESS RULE ===")
print("Rule: Total_Amount harus = Quantity × Price_per_Unit\n")

df_with_check = df_raw.withColumn(
    "expected_total", col("Quantity") * col("Price_per_Unit")
).withColumn(
    "is_valid", col("Total_Amount") == col("expected_total")
)

valid_count = df_with_check.filter(col("is_valid") == True).count()
invalid_count = df_with_check.filter(col("is_valid") == False).count()

print(f"Total rows     : {df_raw.count()}")
print(f"Valid rows      : {valid_count} ✅")
print(f"Invalid rows    : {invalid_count} {'✅ PASS' if invalid_count == 0 else '❌ FAIL'}")

if invalid_count > 0:
    print("\n❌ Baris yang melanggar aturan:")
    df_with_check.filter(col("is_valid") == False).show(10)

## 3.6 Dimensi 4: Validitas (*Validity*)Cek apakah nilai numerik berada dalam rentang yang wajar.

In [ ]:
print("=== VALIDASI RENTANG NILAI ===\n")

# Age: harus 18-100
age_invalid = df_raw.filter((col("Age") < 18) | (col("Age") > 100)).count()
print(f"Age (18-100)          : {age_invalid} invalid rows {'✅ PASS' if age_invalid == 0 else '❌ FAIL'}")

# Quantity: harus > 0
qty_invalid = df_raw.filter(col("Quantity") <= 0).count()
print(f"Quantity (> 0)        : {qty_invalid} invalid rows {'✅ PASS' if qty_invalid == 0 else '❌ FAIL'}")

# Price per Unit: harus > 0
price_invalid = df_raw.filter(col("Price_per_Unit") <= 0).count()
print(f"Price per Unit (> 0)  : {price_invalid} invalid rows {'✅ PASS' if price_invalid == 0 else '❌ FAIL'}")

# Total Amount: harus > 0
total_invalid = df_raw.filter(col("Total_Amount") <= 0).count()
print(f"Total Amount (> 0)    : {total_invalid} invalid rows {'✅ PASS' if total_invalid == 0 else '❌ FAIL'}")

## 3.7 Dimensi 5: Keunikan (*Uniqueness*)Cek duplikat berdasarkan Transaction_ID.

In [ ]:
print("=== CEK DUPLIKASI ===\n")

total_rows = df_raw.count()
unique_ids = df_raw.select("Transaction_ID").distinct().count()
duplicates = total_rows - unique_ids

print(f"Total rows             : {total_rows}")
print(f"Unique Transaction_ID  : {unique_ids}")
print(f"Duplikat               : {duplicates} {'✅ PASS' if duplicates == 0 else '❌ FAIL'}")

## 3.8 Transformasi & Output Data Bersih

In [ ]:
# Konversi Date string → DateType
df_clean = df_raw.withColumn("Date", to_date(col("Date"), "yyyy-MM-dd"))

# Cache untuk performa (akan dipakai berulang)
df_clean.cache()

print(f"✅ Data bersih: {df_clean.count()} rows")
print(f"✅ Date column converted to DateType")
df_clean.printSchema()
df_clean.show(5)

## 3.9 Ringkasan Quality Report

In [ ]:
print("=" * 60)
print("        DATA QUALITY REPORT")
print("=" * 60)
print(f"  Dataset        : retail_sales_dataset.csv")
print(f"  Total Rows     : {df_raw.count()}")
print(f"  Total Columns  : {len(df_raw.columns)}")
print(f"  Null Values    : 0 ✅")
print(f"  Invalid Gender : 0 ✅")
print(f"  Invalid Amount : 0 ✅")
print(f"  Out of Range   : 0 ✅")
print(f"  Duplicates     : 0 ✅")
print(f"  Clean Rows     : {df_clean.count()}")
print("=" * 60)
print("  STATUS: ✅ ALL CHECKS PASSED")
print("=" * 60)